In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/content/drive/MyDrive/comments_merged.csv")

In [3]:
df.head()

,projectID,comment_id,parent_id,depth_level,author_id,author_name,creator_id,is_pledge_master,is_backer,backer_number,...,is_pathfinder,has_children,children_count,text,created_at,likes,phaseLabel,campaignStart,campaignEnd,group
0,5787,1691589,NaN,0,1456332,BlackSiteStudios,1,False,False,NaN,...,False,False,0,WELCOME TO EDEN!\n\nPlease check the FAQ and t...,2025-06-01 23:14:35.600000+00:00,26,Pledge manager - Closed,2025-06-17 14:30:00+00:00,2025-07-19 00:50:35.250000+00:00,0
1,5787,1725920,NaN,0,9482,Yug_the_Juggernaut,0,True,True,1.0,...,False,True,1,@BlackSiteStudios\nCould a person use larger d...,2025-06-17 13:42:18.150000+00:00,0,Pledge manager - Closed,2025-06-17 14:30:00+00:00,2025-07-19 00:50:35.250000+00:00,0
2,5787,1726005,1725920.0,1,1456332,BlackSiteStudios,1,False,False,NaN,...,False,False,0,Sure! The game is designed for a 2x2ft play ar...,2025-06-17 14:09:27.980000+00:00,0,Pledge manager - Closed,2025-06-17 14:30:00+00:00,2025-07-19 00:50:35.250000+00:00,0
3,5787,1725876,NaN,0,137438,Terry_van_Leeuwen,0,True,True,13.0,...,False,False,0,about an hour left to go,2025-06-17 13:26:09.743000+00:00,1,Pledge manager - Closed,2025-06-17 14:30:00+00:00,2025-07-19 00:50:35.250000+00:00,0
4,5787,1725671,NaN,0,12163,Haro4Life,0,False,False,NaN,...,False,False,0,Man 2 hours to go worth staying up to midnight...,2025-06-17 11:42:57.760000+00:00,3,Pledge manager - Closed,2025-06-17 14:30:00+00:00,2025-07-19 00:50:35.250000+00:00,0


In [4]:
df.columns.tolist()

['projectID',
 'comment_id',
 'parent_id',
 'depth_level',
 'author_id',
 'author_name',
 'creator_id',
 'is_pledge_master',
 'is_backer',
 'backer_number',
 'is_prior_backer',
 'is_pathfinder',
 'has_children',
 'children_count',
 'text',
 'created_at',
 'likes',
 'phaseLabel',
 'campaignStart',
 'campaignEnd',
 'group']

In [5]:
len(df)

212988

In [6]:
# 1. 설치
!pip install -U FlagEmbedding bertopic umap-learn hdbscan

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 69.1 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18919 sha256=928d8b45e5e3f832513f241f2f28dd3dd44abe2d1633f0cf95f175297d45118f
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c
  Created wheel for cbor: filename=cbor-1.0.0-cp312-cp312-linux_x86_64.whl size

In [7]:
import numpy as np
import torch
import gdown
import os
from FlagEmbedding import BGEM3FlagModel
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
import gdown

In [8]:
# 2. 원본 보호
data = df.copy()

In [9]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

In [10]:
# 1. thread_df 불러오기
thread_df = pd.read_csv("/content/drive/MyDrive/comments_merged_thread_add_tag.csv")

print(thread_df.columns.tolist())
print(thread_df.head())

['Unnamed: 0', 'projectID', 'thread_id', 'group', 'comment_ids', 'n_comments', 'merged_comment', 'creator_id', 'is_pledge_master', 'is_backer', 'is_prior_backer', 'is_pathfinder']
   Unnamed: 0  projectID  thread_id  group         comment_ids  n_comments  \
0           0       5787    1691589      0           [1691589]           1   
1           1       5787    1725920      0  [1725920, 1726005]           2   
2           2       5787    1725876      0           [1725876]           1   
3           3       5787    1725671      0           [1725671]           1   
4           4       5787    1725512      0           [1725512]           1   

                                      merged_comment  creator_id  \
0  WELCOME TO EDEN!  Please check the FAQ and the...           1   
1  @BlackSiteStudios Could a person use larger di...           1   
2                           about an hour left to go           0   
3  Man 2 hours to go worth staying up to midnight...           0   
4          

In [11]:
thread_df.columns

Index(['Unnamed: 0', 'projectID', 'thread_id', 'group', 'comment_ids',
       'n_comments', 'merged_comment', 'creator_id', 'is_pledge_master',
       'is_backer', 'is_prior_backer', 'is_pathfinder'],
      dtype='object')

In [12]:
sample_thread_df = thread_df.copy().reset_index(drop=True)

sample_thread_df["merged_comment"] = (
    sample_thread_df["merged_comment"]
    .fillna("")
    .astype(str)
    .str.strip()
)

print("빈 merged_comment 수:", (sample_thread_df["merged_comment"] == "").sum())

빈 merged_comment 수: 1


In [13]:
sample_thread_df[
    sample_thread_df["merged_comment"].astype(str).str.strip() == ""
]

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder
21031,21031,222,3196,0,[3196],1,,0,1,1,1,0


In [16]:
thread_df.loc[
    thread_df["merged_comment"].isna() |
    (thread_df["merged_comment"].astype(str).str.strip() == ""),
    ["projectID", "thread_id", "merged_comment"]
]

,projectID,thread_id,merged_comment
21031,222,3196,NaN


In [14]:
texts = sample_thread_df["merged_comment"].tolist()
print("전체 thread_df:", len(thread_df))
print("sample_thread_df:", len(sample_thread_df))
print("texts:", len(texts))
print("프로젝트 수:", sample_thread_df["projectID"].nunique())

전체 thread_df: 73439
sample_thread_df: 73439
texts: 73439
프로젝트 수: 473


In [14]:
print("원본 thread 수:", len(thread_df))
print("NaN/빈 댓글 제거 후:", len(sample_thread_df))
print("제거된 수:", len(thread_df) - len(sample_thread_df))

원본 thread 수: 73439
NaN/빈 댓글 제거 후: 73439
제거된 수: 0


In [15]:
sample_thread_df = sample_thread_df[
    sample_thread_df["merged_comment"] != ""
].copy()

sample_thread_df.reset_index(drop=True, inplace=True)

In [16]:
print("원본 thread 수:", len(thread_df))
print("NaN/빈 댓글 제거 후:", len(sample_thread_df))
print("제거된 수:", len(thread_df) - len(sample_thread_df))

원본 thread 수: 73439
NaN/빈 댓글 제거 후: 73438
제거된 수: 1


빈 thread 1개 제거

In [17]:
texts = sample_thread_df["merged_comment"].tolist()
print("프로젝트 수:", sample_thread_df["projectID"].nunique())

display(sample_thread_df["projectID"].value_counts().head(20))

프로젝트 수: 473


,count
projectID,
3890,2616
8203,1812
222,1778
1927,1612
1032,1486
1926,1469
3176,1466
2758,1449
1318,1449


In [18]:
# =========================
# group 확인 및 보정
# =========================

print("group 컬럼 존재 여부:", "group" in sample_thread_df.columns)

sample_thread_df["group"] = sample_thread_df["group"].fillna(0).astype(int)

print("group 값 분포")
display(sample_thread_df["group"].value_counts().sort_index())

print("프로젝트별 group 개수 분포")
project_group_count = (
    sample_thread_df.groupby("projectID")["group"]
    .nunique()
    .value_counts()
    .sort_index()
)

display(project_group_count)

print("group이 하나만 있는 프로젝트 수:", (sample_thread_df.groupby("projectID")["group"].nunique() == 1).sum())

only_one_group_projects = (
    sample_thread_df.groupby("projectID")["group"]
    .nunique()
    .reset_index(name="n_group")
    .query("n_group == 1")
)

display(only_one_group_projects.head(30))

group 컬럼 존재 여부: True
group 값 분포


,count
group,
0,34571
1,38867


프로젝트별 group 개수 분포


,count
group,
1,45
2,428


group이 하나만 있는 프로젝트 수: 45


,projectID,n_group
69,2081,1
78,2200,1
81,2217,1
87,2242,1
91,2275,1
94,2304,1
115,2519,1
129,2728,1
131,2735,1
141,2816,1


In [23]:
#3. BGE-M3 임베딩 생성
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=True if device == "cuda" else False
)


device: cuda


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [19]:
lengths = thread_df["merged_comment"].astype(str).str.split().str.len()

print(lengths.describe())

count    73439.000000
mean        91.293863
std        174.191273
min          1.000000
25%         19.000000
50%         44.000000
75%         96.000000
max       9875.000000
Name: merged_comment, dtype: float64


In [20]:
sample_thread_df["merged_comment"].map(type).value_counts()

,count
merged_comment,
<class 'str'>,73438


In [26]:
embeddings = model.encode(
    texts,
    batch_size=32,
    max_length=1024
)["dense_vecs"]

embeddings = np.array(embeddings)

print(embeddings.shape)


pre tokenize: 100%|██████████| 2295/2295 [00:29<00:00, 77.29it/s]

Inference Embeddings: 100%|██████████| 2295/2295 [06:10<00:00,  6.19it/s]


(73438, 1024)


In [28]:
np.save(
    "/content/drive/MyDrive/bge_m3_embeddings.npy",
    embeddings
)

In [22]:
embeddings = np.load("/content/drive/MyDrive/bge_m3_embeddings.npy")

In [23]:
# 4. 모델 설정
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
    low_memory=True
)

hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    min_samples=3,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model_final = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

In [24]:
# 모델 실행
topics, probs = topic_model_final.fit_transform(
    texts,
    embeddings
)

print("BERTopic 완료")
print("토픽 개수:", len(set(topics)))
print("-1 개수:", sum(np.array(topics) == -1))
print("-1 비율:", round((np.array(topics) == -1).mean() * 100, 2), "%")

2026-06-09 04:23:33,026 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-09 04:26:10,735 - BERTopic - Dimensionality - Completed ✓
2026-06-09 04:26:10,740 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-09 04:26:23,670 - BERTopic - Cluster - Completed ✓
2026-06-09 04:26:23,691 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-09 04:26:28,500 - BERTopic - Representation - Completed ✓


BERTopic 완료
토픽 개수: 193
-1 개수: 31001
-1 비율: 42.21 %


In [40]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
    low_memory=True
)

hdbscan_model = HDBSCAN(
    min_cluster_size=30,
    min_samples=1,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model_final_1 = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    language="multilingual",
    calculate_probabilities=False,
    verbose=True
)

topics, probs = topic_model_final_1.fit_transform(
    texts,
    embeddings
)

print("BERTopic 완료")
print("토픽 개수:", len(set(topics)))
print("-1 개수:", sum(np.array(topics) == -1))
print("-1 비율:", round((np.array(topics) == -1).mean() * 100, 2), "%")

2026-06-09 04:38:20,468 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-09 04:40:34,656 - BERTopic - Dimensionality - Completed ✓
2026-06-09 04:40:34,660 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-09 04:40:47,115 - BERTopic - Cluster - Completed ✓
2026-06-09 04:40:47,136 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-09 04:40:53,261 - BERTopic - Representation - Completed ✓


BERTopic 완료
토픽 개수: 365
-1 개수: 27948
-1 비율: 38.06 %


In [25]:
topics_original_final = topics.copy()
np.save("/content/drive/MyDrive/topics_original_final_merged.npy", topics_original_final)

sample_thread_df["topic_original"] = topics_original_final
sample_thread_df["was_outlier"] = sample_thread_df["topic_original"] == -1

sample_thread_df.to_pickle("/content/drive/MyDrive/thread_df_topics_original_final_merged.pkl")

In [26]:
topics_original_info = topic_model_final.get_topic_info()
topics_original_info.head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,31001,-1_the_to_and_of,"[the, to, and, of, it, you, in, that, for, is]","[As a total stalker-lore fan, who discovered v..."
1,0,2151,0_solo_mode_player_players,"[solo, mode, player, players, play, coop, co, ...","[Can we get a solo mode? :D, +1 for a solo mod..."
2,1,1884,1_excited_game_wait_this,"[excited, game, wait, this, looks, forward, th...",[This game looks awesome. Can't wait to get i...
3,2,1166,2_dc_batman_marvel_united,"[dc, batman, marvel, united, villains, charact...",[This will upset people but I'm Only here for ...
4,3,1102,3_shipping_eu_china_cost,"[shipping, eu, china, cost, costs, australia, ...",[Why for the love of everything holy is shippi...


In [27]:
topics_original_info = topic_model_final.get_topic_info()

topics_original_info.to_csv(
    "/content/drive/MyDrive/topics_original_info_merged.csv",
    index=False
)

In [28]:
# outlier 처리
topics_reassigned_final = topic_model_final.reduce_outliers(
    texts,
    topics_original_final,
    embeddings=embeddings,
    strategy="embeddings"
)

topics_reassigned_final = np.array(topics_reassigned_final)

sample_thread_df["topic_reassigned"] = topics_reassigned_final

np.save("/content/drive/MyDrive/topics_reassigned_final_merged.npy", topics_reassigned_final)

sample_thread_df.to_pickle("/content/drive/MyDrive/thread_df_topics_original_reassigned_final_merged.pkl")

print("재할당 후 -1 비율:", round((topics_reassigned_final == -1).mean() * 100, 2), "%")

재할당 후 -1 비율: 0.0 %


embedding similarity 기준으로 기존 토픽에 재할당 후 토픽 수 변화 비교

In [29]:
# 1. 토픽 의미/키워드 확인용
topic_info_original = topic_model_final.get_topic_info()

# 2. 재할당 후 최종 문서 수 확인용
topic_counts_final = (
    pd.Series(topics_reassigned_final)
    .value_counts()
    .reset_index()
)

topic_counts_final.columns = ["Topic", "Count_final"]

# 3. 둘 합치기
topic_info_final_view = topic_info_original.merge(
    topic_counts_final,
    on="Topic",
    how="left"
)

topic_info_final_view["Count_final"] = (
    topic_info_final_view["Count_final"]
    .fillna(0)
    .astype(int)
)

topic_info_final_view.head(20)

,Topic,Count,Name,Representation,Representative_Docs,Count_final
0,-1,31001,-1_the_to_and_of,"[the, to, and, of, it, you, in, that, for, is]","[As a total stalker-lore fan, who discovered v...",0
1,0,2151,0_solo_mode_player_players,"[solo, mode, player, players, play, coop, co, ...","[Can we get a solo mode? :D, +1 for a solo mod...",2326
2,1,1884,1_excited_game_wait_this,"[excited, game, wait, this, looks, forward, th...",[This game looks awesome. Can't wait to get i...,2971
3,2,1166,2_dc_batman_marvel_united,"[dc, batman, marvel, united, villains, charact...",[This will upset people but I'm Only here for ...,1548
4,3,1102,3_shipping_eu_china_cost,"[shipping, eu, china, cost, costs, australia, ...",[Why for the love of everything holy is shippi...,1541
5,4,1011,4_price_expensive_game_base,"[price, expensive, game, base, cost, this, is,...",[1. Lower the price 2. offer a less expensive ...,1842
6,5,936,5_stretch_goals_goal_daily,"[stretch, goals, goal, daily, unlocked, unlock...",[Will there be stretch goals or what we see is...,1092
7,6,864,6_playmat_mat_neoprene_mats,"[playmat, mat, neoprene, mats, playmats, board...",[Interesting to see the game mat being so expe...,995
8,7,863,7_sleeves_sleeved_sleeve_cards,"[sleeves, sleeved, sleeve, cards, fit, insert,...",[Will you offer card sleeves? And does the in...,915
9,8,832,8_launch_date_when_start,"[launch, date, when, start, campaign, live, se...",[When?|We didn't share the launch date yet. So...,1072


In [30]:
# 재할당 후 크게 늘어난 토픽
topic_info_final_view["increase"] = (
    topic_info_final_view["Count_final"] - topic_info_final_view["Count"]
)

topic_info_final_view["increase_ratio"] = (
    topic_info_final_view["Count_final"] / topic_info_final_view["Count"]
)

topic_info_final_view[
    ["Topic", "Count", "Count_final", "increase", "increase_ratio", "Name", "Representation"]
].sort_values("increase", ascending=False).head(20)

,Topic,Count,Count_final,increase,increase_ratio,Name,Representation
2,1,1884,2971,1087,1.576964,1_excited_game_wait_this,"[excited, game, wait, this, looks, forward, th..."
5,4,1011,1842,831,1.821958,4_price_expensive_game_base,"[price, expensive, game, base, cost, this, is,..."
74,73,152,952,800,6.263158,73_cards_faction_new_art,"[cards, faction, new, art, player, expansion, ..."
44,43,240,1033,793,4.304167,43_boss_difficulty_attack_action,"[boss, difficulty, attack, action, you, combat..."
12,11,751,1502,751,2.000000,11_pledge_all_add_apocalypse,"[pledge, all, add, apocalypse, ons, in, new, e..."
11,10,758,1477,719,1.948549,10_box_boxes_big_storage,"[box, boxes, big, storage, fit, solution, ever..."
10,9,813,1403,590,1.725707,9_tiles_boards_board_tile,"[tiles, boards, board, tile, player, layer, du..."
82,81,139,729,590,5.244604,81_available_campaign_base_previous,"[available, campaign, base, previous, first, w..."
75,74,152,699,547,4.598684,74_projects_communication_company_project,"[projects, communication, company, project, ga..."
137,136,80,619,539,7.737500,136_skyrim_eso_game_elder,"[skyrim, eso, game, elder, scrolls, rpg, like,..."


In [31]:
# 재할당으로 새로 붙은 댓글만 확인
sample_thread_df["topic_original"] = topics_original_final
sample_thread_df["topic_final"] = topics_reassigned_final
sample_thread_df["was_reassigned"] = (
    sample_thread_df["topic_original"] != sample_thread_df["topic_final"]
)

check_topics = [10, 1, 52, 8, 203, 85, 19, 49, 107, 55]

for tid in check_topics:
    print("=" * 80)
    print(f"Topic {tid}")

    temp = sample_thread_df[
        (sample_thread_df["topic_final"] == tid) &
        (sample_thread_df["was_reassigned"])
    ].copy()

    print("재할당으로 새로 붙은 thread 수:", len(temp))

    if len(temp) == 0:
        print("재할당된 thread 없음")
        continue

    display(
        temp[
            [
                "thread_id",
                "projectID",
                "group",
                "merged_comment",
                "topic_original",
                "topic_final"
            ]
        ].sample(
            n=min(5, len(temp)),
            random_state=42
        )
    )

Topic 10
재할당으로 새로 붙은 thread 수: 719


,thread_id,projectID,group,merged_comment,topic_original,topic_final
17066,206452,1926,0,Definitely all the expansions included in the ...,-1,10
25608,445285,2635,0,Sorry if this was asked and I missed it. I am ...,-1,10
5634,1050056,4065,0,Any idea if the Folded Space insert for the or...,-1,10
57355,90640,1078,1,What's the difference between the godly pledge...,-1,10
27839,813723,3830,0,will the new box be available separately?|A we...,-1,10


Topic 1
재할당으로 새로 붙은 thread 수: 1087


,thread_id,projectID,group,merged_comment,topic_original,topic_final
41386,899993,2969,1,I look forward to play Feudum! I was at work w...,-1,1
2016,1583950,5386,0,"Hi AR, the game looks very cool! I'm a big fan...",-1,1
34314,1921456,6381,0,"I saw this game yesterday, and while the idea ...",-1,1
53474,127531,1114,1,Update is live everyone! Also - thanks sooo mu...,-1,1
39150,784193,2225,1,Appena effettuato il pladge eternal. Complimen...,-1,1


Topic 52
재할당으로 새로 붙은 thread 수: 88


...

,thread_id,projectID,group,merged_comment,topic_original,topic_final
9359,830621,3592,0,This looks exciting! Can’t wait to receive Mon...,-1,49
43554,868958,3818,1,"They say....""Goal reached in 2 hours and 41 mi...",-1,49
1557,596367,3723,0,So NA backers are still months away from recei...,-1,49
6962,651997,3176,0,when can we expect the localization for German...,-1,49
3647,160496,4590,0,Check the KS comments for updates on placed an...,-1,49


Topic 107
재할당으로 새로 붙은 thread 수: 288


,thread_id,projectID,group,merged_comment,topic_original,topic_final
11460,562461,2757,0,@AR_Jordan — in the One Stop Co-Op Shop (truly...,-1,107
44158,789745,3383,1,Still on the fence on whether to go for just t...,-1,107
65817,741184,3582,1,I have the 1st edition and have a few question...,-1,107
10690,395285,2264,0,I saw that there's 22 ship boards. Does that ...,-1,107
48974,422482,2295,1,the requirement for a sorcerer is to have a ma...,-1,107


Topic 55
재할당으로 새로 붙은 thread 수: 404


,thread_id,projectID,group,merged_comment,topic_original,topic_final
10365,731447,3554,0,#originalbackers Pourrait on avoir un update ...,-1,55
36831,723870,3278,1,"Bonjour, ça a sûrement déjà été précisé mais j...",-1,55
70081,64902,954,1,"Hi ! I do not find the info, but maybe i misse...",-1,55
4067,691633,3539,0,"Por el correo que me has enviado, vuelvo a seg...",-1,55
6180,1084285,3748,0,Hi all - Really nice to see that the cats will...,-1,55


In [43]:
sample_thread_df.columns

Index(['Unnamed: 0', 'projectID', 'thread_id', 'group', 'comment_ids',
       'n_comments', 'merged_comment', 'creator_id', 'is_pledge_master',
       'is_backer', 'is_prior_backer', 'is_pathfinder', 'topic_original',
       'was_outlier', 'topic_reassigned', 'topic_final', 'was_reassigned'],
      dtype='object')

In [44]:
print("topics:", len(topics))

try:
    print("new_topics:", len(new_topics))
except:
    print("new_topics 없음")

topics: 73438
new_topics 없음


In [33]:
topic_model_final.get_topic_info().head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,31001,-1_the_to_and_of,"[the, to, and, of, it, you, in, that, for, is]","[As a total stalker-lore fan, who discovered v..."
1,0,2151,0_solo_mode_player_players,"[solo, mode, player, players, play, coop, co, ...","[Can we get a solo mode? :D, +1 for a solo mod..."
2,1,1884,1_excited_game_wait_this,"[excited, game, wait, this, looks, forward, th...",[This game looks awesome. Can't wait to get i...
3,2,1166,2_dc_batman_marvel_united,"[dc, batman, marvel, united, villains, charact...",[This will upset people but I'm Only here for ...
4,3,1102,3_shipping_eu_china_cost,"[shipping, eu, china, cost, costs, australia, ...",[Why for the love of everything holy is shippi...


In [34]:
# 5. BERTopic 결과 저장
sample_thread_df.to_csv(
    "comment_bertopic_thread_result.csv",
    index=False,
    encoding="utf-8-sig"
)

topic_info = topic_model_final.get_topic_info()

topic_info.to_csv(
    "bertopic_topic_info.csv",
    index=False,
    encoding="utf-8-sig"
)

display(topic_info.head(20))

,Topic,Count,Name,Representation,Representative_Docs
0,-1,31001,-1_the_to_and_of,"[the, to, and, of, it, you, in, that, for, is]","[As a total stalker-lore fan, who discovered v..."
1,0,2151,0_solo_mode_player_players,"[solo, mode, player, players, play, coop, co, ...","[Can we get a solo mode? :D, +1 for a solo mod..."
2,1,1884,1_excited_game_wait_this,"[excited, game, wait, this, looks, forward, th...",[This game looks awesome. Can't wait to get i...
3,2,1166,2_dc_batman_marvel_united,"[dc, batman, marvel, united, villains, charact...",[This will upset people but I'm Only here for ...
4,3,1102,3_shipping_eu_china_cost,"[shipping, eu, china, cost, costs, australia, ...",[Why for the love of everything holy is shippi...
5,4,1011,4_price_expensive_game_base,"[price, expensive, game, base, cost, this, is,...",[1. Lower the price 2. offer a less expensive ...
6,5,936,5_stretch_goals_goal_daily,"[stretch, goals, goal, daily, unlocked, unlock...",[Will there be stretch goals or what we see is...
7,6,864,6_playmat_mat_neoprene_mats,"[playmat, mat, neoprene, mats, playmats, board...",[Interesting to see the game mat being so expe...
8,7,863,7_sleeves_sleeved_sleeve_cards,"[sleeves, sleeved, sleeve, cards, fit, insert,...",[Will you offer card sleeves? And does the in...
9,8,832,8_launch_date_when_start,"[launch, date, when, start, campaign, live, se...",[When?|We didn't share the launch date yet. So...


BERTopic-hierarchical_topics(): BERTopic이 토픽 간 유사도 계산하여 계층구조 생성

In [35]:
hierarchical_topics = topic_model_final.hierarchical_topics(texts)

100%|██████████| 191/191 [00:01<00:00, 159.18it/s]


In [36]:
topic_model_final.visualize_hierarchy()

In [37]:
hierarchical_topics.head()
print(hierarchical_topics.columns.tolist())

['Parent_ID', 'Parent_Name', 'Topics', 'Child_Left_ID', 'Child_Left_Name', 'Child_Right_ID', 'Child_Right_Name', 'Distance']


In [45]:
sample_thread_df["topic_original"] = sample_thread_df["topic_original"]
sample_thread_df["topic_final"] = sample_thread_df["topic_final"]

In [46]:
print("전체 행 수:", len(sample_thread_df))
print("topics 길이:", len(topics))
print("topic_final 결측:", sample_thread_df["topic_final"].isna().sum())

print("original -1 개수:", (sample_thread_df["topic_original"] == -1).sum())
print("final -1 개수:", (sample_thread_df["topic_final"] == -1).sum())
print("final -1 비율:", round((sample_thread_df["topic_final"] == -1).mean() * 100, 2), "%")

전체 행 수: 73438
topics 길이: 73438
topic_final 결측: 0
original -1 개수: 27948
final -1 개수: 0
final -1 비율: 0.0 %


In [47]:
sample_thread_df.to_csv(
    "comment_bertopic_thread_result_with_outlier_reduction.csv",
    index=False,
    encoding="utf-8-sig"
)

topic 20개로 압축 후 병합

In [48]:
# 1. hierarchy 복사

hier_df_20 = hierarchical_topics.copy()

def parse_topics_20(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

hier_df_20["Topics_parsed"] = hier_df_20["Topics"].apply(parse_topics_20)
hier_df_20["Topics_set"] = hier_df_20["Topics_parsed"].apply(lambda x: frozenset(x))


# 2. 원본 토픽 목록 준비

base_topics_20 = sorted(
    topic_info_final_view[
        topic_info_final_view["Topic"] != -1
    ]["Topic"].tolist()
)

clusters_20 = [frozenset([t]) for t in base_topics_20]


# 3. BERTopic hierarchy 기준으로 20개가 될 때까지 병합

hier_df_sorted_20 = hier_df_20.sort_values("Distance", ascending=True)

for _, row in hier_df_sorted_20.iterrows():
    parent_set = row["Topics_set"]

    child_clusters = [
        c for c in clusters_20
        if c.issubset(parent_set)
    ]

    union_set = (
        frozenset().union(*child_clusters)
        if child_clusters
        else frozenset()
    )

    if union_set == parent_set and len(child_clusters) >= 2:
        clusters_20 = [
            c for c in clusters_20
            if c not in child_clusters
        ]

        clusters_20.append(parent_set)

    if len(clusters_20) == 20:
        break

print("현재 cluster 개수:", len(clusters_20))


# 4. 20개 MetaTopic 매핑표 생성

meta_rows_20 = []

for i, topic_set in enumerate(clusters_20, start=1):
    for topic in topic_set:
        meta_rows_20.append({
            "MetaTopic_20": i,
            "Topic": topic
        })

topic_meta_map_20 = pd.DataFrame(meta_rows_20)


# 5. topic_info_final_view와 병합

topic_meta_view_20 = topic_info_final_view.merge(
    topic_meta_map_20,
    on="Topic",
    how="left"
)

topic_meta_view_20[
    ["MetaTopic_20", "Topic", "Count_final", "Name", "Representation"]
].sort_values(
    ["MetaTopic_20", "Count_final"],
    ascending=[True, False]
)

현재 cluster 개수: 20


,MetaTopic_20,Topic,Count_final,Name,Representation
71,1.0,70,333,70_polish_language_version_poland,"[polish, language, version, poland, we, transl..."
83,1.0,82,209,82_polish_language_version_poland,"[polish, language, version, poland, please, ch..."
19,2.0,18,647,18_italian_language_please_italiano,"[italian, language, please, italiano, translat..."
37,2.0,36,474,36_italian_language_translation_languages,"[italian, language, translation, languages, it..."
166,3.0,165,212,165_backer_number_got_congrats,"[backer, number, got, congrats, seconds, damn,..."
...,...,...,...,...,...
142,20.0,141,88,141_mystica_terra_innovation_gaia,"[mystica, terra, innovation, gaia, tm, age, ao..."
170,20.0,169,87,169_vindication_vestige_neb_small,"[vindication, vestige, neb, small, oneb, compa..."
173,20.0,172,85,172_assent_stormsunder_wild_hunt,"[assent, stormsunder, wild, hunt, rising, lavo..."
187,20.0,186,80,186_1643_monolith_claustrophobia_demons,"[1643, monolith, claustrophobia, demons, style..."


In [49]:
for mt in sorted(topic_meta_view_20["MetaTopic_20"].dropna().unique()):
    print("=" * 80)
    print(f"MetaTopic {mt}")

    display(
        topic_meta_view_20[
            topic_meta_view_20["MetaTopic_20"] == mt
        ][
            ["Topic", "Count_final", "Name", "Representation"]
        ].sort_values("Count_final", ascending=False).head(15)
    )

MetaTopic 1.0


,Topic,Count_final,Name,Representation
71,70,333,70_polish_language_version_poland,"[polish, language, version, poland, we, transl..."
83,82,209,82_polish_language_version_poland,"[polish, language, version, poland, please, ch..."


MetaTopic 2.0


,Topic,Count_final,Name,Representation
19,18,647,18_italian_language_please_italiano,"[italian, language, please, italiano, translat..."
37,36,474,36_italian_language_translation_languages,"[italian, language, translation, languages, it..."


MetaTopic 3.0


...

,Topic,Count_final,Name,Representation
18,17,848,17_free_pledge_48_bird,"[free, pledge, 48, bird, early, gift, hours, f..."
20,19,555,19_pay_stretch_stretchpay_payments,"[pay, stretch, stretchpay, payments, payment, ..."
32,31,476,31_access_pledge_manager_pm,"[access, pledge, manager, pm, gamefound, late,..."
87,86,275,86_cancel_charged_pledge_paypal,"[cancel, charged, pledge, paypal, payment, edi..."
109,108,273,108_error_pledge_my_fixed,"[error, pledge, my, fixed, try, bug, add, issu..."
140,139,243,139_morning_tomorrow_sleep_night,"[morning, tomorrow, sleep, night, questions, d..."
134,133,178,133_pledge_upgrade_manager_pm,"[pledge, upgrade, manager, pm, change, yes, la..."
88,87,170,87_paypal_payment_credit_bank,"[paypal, payment, credit, bank, card, gamefoun..."
159,158,150,158_late_pledge_yes_pledges,"[late, pledge, yes, pledges, manager, will, ca..."
145,144,126,144_open_manager_pm_long,"[open, manager, pm, long, late, months, pledge..."


MetaTopic 19.0


,Topic,Count_final,Name,Representation
26,25,618,25_funded_congrats_congratulations_thank,"[funded, congrats, congratulations, thank, fun..."
64,63,549,63_banned_pentaquark_handsome_books,"[banned, pentaquark, handsome, books, shy, tur..."
45,44,389,44_money_take_wallet_my,"[money, take, wallet, my, shut, up, ready, wal..."
118,117,338,117_fog_countdown_coming_almost,"[fog, countdown, coming, almost, final, close,..."
59,58,289,58_minutes_hours_go_hour,"[minutes, hours, go, hour, left, mins, days, m..."
129,128,282,128_typo_typos_spelling_page,"[typo, typos, spelling, page, spelled, says, m..."
189,188,251,188_arrived_hypeeee_first_pokładzie,"[arrived, hypeeee, first, pokładzie, jestem, c..."
58,57,215,57_vfi_asia_south_africa,"[vfi, asia, south, africa, shipping, using, in..."
117,116,205,116_followers_backers_1000_000,"[followers, backers, 1000, 000, follower, wow,..."
72,71,200,71_brazil_ship_shipping_brasil,"[brazil, ship, shipping, brasil, brazilian, nã..."


MetaTopic 20.0


,Topic,Count_final,Name,Representation
1,0,2326,0_solo_mode_player_players,"[solo, mode, player, players, play, coop, co, ..."
5,4,1842,4_price_expensive_game_base,"[price, expensive, game, base, cost, this, is,..."
3,2,1548,2_dc_batman_marvel_united,"[dc, batman, marvel, united, villains, charact..."
44,43,1033,43_boss_difficulty_attack_action,"[boss, difficulty, attack, action, you, combat..."
74,73,952,73_cards_faction_new_art,"[cards, faction, new, art, player, expansion, ..."
13,12,852,12_standees_minis_standee_acrylic,"[standees, minis, standee, acrylic, miniatures..."
75,74,699,74_projects_communication_company_project,"[projects, communication, company, project, ga..."
24,23,692,23_gamefound_kickstarter_comments_platform,"[gamefound, kickstarter, comments, platform, k..."
16,15,669,15_sundrop_minis_paint_sundropped,"[sundrop, minis, paint, sundropped, wash, opti..."
28,27,663,27_scenarios_story_scenario_each,"[scenarios, story, scenario, each, hours, play..."


In [50]:
# =========================
# 2. 7개 최종 카테고리 생성
# =========================

topic_category_df = topic_meta_view_20.copy()

topic_category_df["FinalCategory"] = "Other"

In [51]:
# =========================
# 3. MetaTopic 기준 1차 분류
# =========================

meta_to_final_category = {
    # 1. Language & Localization
    1: "Language & Localization",    # Polish
    2: "Language & Localization",    # Italian
    4: "Language & Localization",    # Spanish
    5: "Language & Localization",    # German
    6: "Language & Localization",    # German / Essen
    16: "Language & Localization",   # French / English / language

    # 2. Community Reaction
    3: "Community Reaction",         # backer number / congrats
    7: "Community Reaction",         # hype / let's go
    9: "Community Reaction",         # excited / wait / welcome
    14: "Community Reaction",        # looks / art / cool

    # 3. Components & Production
    8: "Components & Production",    # tiles / boards / tokens
    12: "Components & Production",   # dice / chip / decks / CTG-specific components
    13: "Components & Production",   # box / playmat / sleeves / storage

    # 4. Shipping / Price / Tax
    11: "Shipping / Price / Tax",    # shipping / VAT / price

    # 5. Campaign / Pledge
    15: "Campaign / Pledge",         # pledge / stretch goals / add-ons / retail
    18: "Campaign / Pledge",         # pledge manager / payment / late pledge

    # 6. Schedule / Updates
    10: "Schedule / Updates",        # launch / delivery / update / tracking

    # 7. Gameplay / Rules / Content
    17: "Gameplay / Rules / Content", # rulebook / video / TTS / app
    20: "Gameplay / Rules / Content"  # game-specific titles / content
}

topic_category_df["FinalCategory"] = (
    topic_category_df["MetaTopic_20"]
    .map(meta_to_final_category)
    .fillna("Other")
)

In [52]:
# =========================
# 4. 토픽 단위 수동 재분류
# =========================
# MetaTopic으로 크게 묶은 뒤, 이상하게 들어간 토픽만 여기서 덮어쓰기

manual_topic_to_final_category = {
    # 예시: price가 campaign 쪽에 들어갔으면 비용 쪽으로 이동
    4: "Shipping / Price / Tax",       # price_expensive_game_base

    # solo mode는 gameplay
    0: "Gameplay / Rules / Content",   # solo_mode_player_players

    # DC/Batman/Marvel 같은 IP/캐릭터 토픽은 content
    2: "Gameplay / Rules / Content",   # dc_batman_marvel_united

    # standees/minis는 구성품
    12: "Components & Production",     # standees_minis_standee_acrylic
    15: "Components & Production",     # sundrop_minis_paint_sundropped

    # delivery date는 일정/업데이트
    13: "Schedule / Updates",          # delivery_date_2025_2024

    # launch date도 일정/업데이트
    8: "Schedule / Updates",           # launch_date_when_start

    # VAT는 비용/배송/세금
    14: "Shipping / Price / Tax",      # vat_eu_tax_pay

    # shipping은 비용/배송/세금
    3: "Shipping / Price / Tax",       # shipping_eu_china_cost
}

for topic, category in manual_topic_to_final_category.items():
    topic_category_df.loc[
        topic_category_df["Topic"] == topic,
        "FinalCategory"
    ] = category

In [53]:
# =========================
# 5. 최종 확인
# =========================

display(
    topic_category_df[
        ["MetaTopic_20", "Topic", "Count_final", "Name", "FinalCategory"]
    ]
    .sort_values(
        ["FinalCategory", "Count_final"],
        ascending=[True, False]
    )
)

print("카테고리별 토픽 수")
display(topic_category_df["FinalCategory"].value_counts())

print("카테고리별 댓글 수")
display(
    topic_category_df
    .groupby("FinalCategory")["Count_final"]
    .sum()
    .sort_values(ascending=False)
)

,MetaTopic_20,Topic,Count_final,Name,FinalCategory
12,15.0,11,1502,11_pledge_all_add_apocalypse,Campaign / Pledge
6,15.0,5,1092,5_stretch_goals_goal_daily,Campaign / Pledge
18,18.0,17,848,17_free_pledge_48_bird,Campaign / Pledge
82,15.0,81,729,81_available_campaign_base_previous,Campaign / Pledge
39,15.0,38,704,38_retail_exclusive_available_gamefound,Campaign / Pledge
...,...,...,...,...,...
34,11.0,33,373,33_canada_shipping_canadian_friendly,Shipping / Price / Tax
89,11.0,88,247,88_price_expensive_insane_prices,Shipping / Price / Tax
188,11.0,187,243,187_vat_price_retail_ps5,Shipping / Price / Tax
143,11.0,142,137,142_price_pricing_much_cost,Shipping / Price / Tax


카테고리별 토픽 수


,count
FinalCategory,
Gameplay / Rules / Content,51
Campaign / Pledge,32
Components & Production,28
Language & Localization,23
Other,22
Community Reaction,17
Schedule / Updates,11
Shipping / Price / Tax,9


카테고리별 댓글 수


,Count_final
FinalCategory,
Gameplay / Rules / Content,18895
Campaign / Pledge,11970
Components & Production,10994
Language & Localization,8310
Community Reaction,8277
Shipping / Price / Tax,5992
Other,4886
Schedule / Updates,4114


In [54]:
# 누락된 토픽 확인
unassigned_topics = topic_category_df[
    (topic_category_df["Topic"] != -1) &
    (
        (topic_category_df["FinalCategory"].isna()) |
        (topic_category_df["FinalCategory"] == "Other")
    )
][
    [
        "Topic",
        "Count_final",
        "Name",
        "Representation",
        "MetaTopic_20"
    ]
].sort_values(
    "Count_final",
    ascending=False
)

display(unassigned_topics)

,Topic,Count_final,Name,Representation,MetaTopic_20
26,25,618,25_funded_congrats_congratulations_thank,"[funded, congrats, congratulations, thank, fun...",19.0
64,63,549,63_banned_pentaquark_handsome_books,"[banned, pentaquark, handsome, books, shy, tur...",19.0
45,44,389,44_money_take_wallet_my,"[money, take, wallet, my, shut, up, ready, wal...",19.0
118,117,338,117_fog_countdown_coming_almost,"[fog, countdown, coming, almost, final, close,...",19.0
59,58,289,58_minutes_hours_go_hour,"[minutes, hours, go, hour, left, mins, days, m...",19.0
129,128,282,128_typo_typos_spelling_page,"[typo, typos, spelling, page, spelled, says, m...",19.0
189,188,251,188_arrived_hypeeee_first_pokładzie,"[arrived, hypeeee, first, pokładzie, jestem, c...",19.0
58,57,215,57_vfi_asia_south_africa,"[vfi, asia, south, africa, shipping, using, in...",19.0
117,116,205,116_followers_backers_1000_000,"[followers, backers, 1000, 000, follower, wow,...",19.0
72,71,200,71_brazil_ship_shipping_brasil,"[brazil, ship, shipping, brasil, brazilian, nã...",19.0


In [55]:
# =========================
# MetaTopic 19 수동 재분류
# =========================

manual_topic_to_final_category.update({
    # Community Reaction
    25: "Community Reaction",   # funded / congrats
    44: "Community Reaction",   # money / wallet / shut up and take my money
    117: "Community Reaction",  # countdown / coming / almost
    58: "Community Reaction",   # minutes / hours left
    188: "Community Reaction",  # arrived / hype
    116: "Community Reaction",  # followers / backers milestone
    121: "Community Reaction",  # woot / all-in
    63: "Community Reaction",   # meme-like / off-topic excitement

    # Shipping / Price / Tax
    57: "Shipping / Price / Tax",   # Asia / South Africa shipping
    71: "Shipping / Price / Tax",   # Brazil shipping
    157: "Shipping / Price / Tax",  # Mexico / South America shipping

    # Language & Localization
    97: "Language & Localization",   # Czech / Slovak
    112: "Language & Localization",  # Chinese version
    137: "Language & Localization",  # Ukrainian localization
    151: "Language & Localization",  # French text / players

    # Gameplay / Rules / Content
    161: "Gameplay / Rules / Content",  # Realms Awaken / project-specific
    124: "Gameplay / Rules / Content",  # war / nations / theme
    103: "Gameplay / Rules / Content",  # Lords of Hellas / Ragnarok
    178: "Gameplay / Rules / Content",  # BGG page / boardgame info
    131: "Gameplay / Rules / Content",  # Russia / Ukraine / war theme

    # Schedule / Updates
    128: "Schedule / Updates",  # typo / spelling / page correction
})

In [56]:
for topic, category in manual_topic_to_final_category.items():
    topic_category_df.loc[
        topic_category_df["Topic"] == topic,
        "FinalCategory"
    ] = category

In [57]:
print("카테고리별 토픽 수")
display(topic_category_df["FinalCategory"].value_counts())

print("카테고리별 댓글 수")
display(
    topic_category_df
    .groupby("FinalCategory")["Count_final"]
    .sum()
    .sort_values(ascending=False)
)

print("Other 남은 토픽")
display(
    topic_category_df[
        topic_category_df["FinalCategory"] == "Other"
    ][["Topic", "Count_final", "Name", "Representation", "MetaTopic_20"]]
    .sort_values("Count_final", ascending=False)
)

카테고리별 토픽 수


,count
FinalCategory,
Gameplay / Rules / Content,56
Campaign / Pledge,32
Components & Production,28
Language & Localization,27
Community Reaction,25
Shipping / Price / Tax,12
Schedule / Updates,12
Other,1


카테고리별 댓글 수


,Count_final
FinalCategory,
Gameplay / Rules / Content,19604
Campaign / Pledge,11970
Community Reaction,11110
Components & Production,10994
Language & Localization,8772
Shipping / Price / Tax,6592
Schedule / Updates,4396
Other,0


Other 남은 토픽


,Topic,Count_final,Name,Representation,MetaTopic_20
0,-1,0,-1_the_to_and_of,"[the, to, and, of, it, you, in, that, for, is]",NaN


In [58]:
topic_category_df = topic_category_df[
    topic_category_df["Topic"] != -1
].copy()

In [59]:
print("카테고리별 토픽 수")
display(topic_category_df["FinalCategory"].value_counts())

print("카테고리별 댓글 수")
display(
    topic_category_df
    .groupby("FinalCategory")["Count_final"]
    .sum()
    .sort_values(ascending=False)
)

카테고리별 토픽 수


,count
FinalCategory,
Gameplay / Rules / Content,56
Campaign / Pledge,32
Components & Production,28
Language & Localization,27
Community Reaction,25
Shipping / Price / Tax,12
Schedule / Updates,12


카테고리별 댓글 수


,Count_final
FinalCategory,
Gameplay / Rules / Content,19604
Campaign / Pledge,11970
Community Reaction,11110
Components & Production,10994
Language & Localization,8772
Shipping / Price / Tax,6592
Schedule / Updates,4396


In [60]:
comment_final_df = sample_thread_df.merge(
    topic_category_df[["Topic", "FinalCategory"]],
    left_on="topic_final",
    right_on="Topic",
    how="left"
).drop(columns=["Topic"])

print("원본 행 수:", len(sample_thread_df))
print("병합 후 행 수:", len(comment_final_df))
print("FinalCategory 결측 수:", comment_final_df["FinalCategory"].isna().sum())

display(comment_final_df["FinalCategory"].value_counts(dropna=False))

원본 행 수: 73438
병합 후 행 수: 73438
FinalCategory 결측 수: 0


,count
FinalCategory,
Gameplay / Rules / Content,19604
Campaign / Pledge,11970
Community Reaction,11110
Components & Production,10994
Language & Localization,8772
Shipping / Price / Tax,6592
Schedule / Updates,4396


In [61]:
comment_final_df.head()

,Unnamed: 0,projectID,thread_id,group,comment_ids,n_comments,merged_comment,creator_id,is_pledge_master,is_backer,is_prior_backer,is_pathfinder,topic_original,was_outlier,topic_reassigned,topic_final,was_reassigned,FinalCategory
0,0,5787,1691589,0,[1691589],1,WELCOME TO EDEN! Please check the FAQ and the...,1,0,0,0,0,321,True,21,21,True,Gameplay / Rules / Content
1,1,5787,1725920,0,"[1725920, 1726005]",2,@BlackSiteStudios Could a person use larger di...,1,1,1,0,0,100,False,101,101,False,Gameplay / Rules / Content
2,2,5787,1725876,0,[1725876],1,about an hour left to go,0,1,1,1,0,39,False,58,58,False,Community Reaction
3,3,5787,1725671,0,[1725671],1,Man 2 hours to go worth staying up to midnight...,0,0,0,0,0,39,False,58,58,False,Community Reaction
4,4,5787,1725512,0,[1725512],1,5 hours to go! Hold on to your butts!,0,0,1,1,0,39,False,58,58,False,Community Reaction


In [62]:
# 카테고리 붙인 상태 저장
sample_thread_df.to_csv(
    "comment_bertopic_thread_result.csv",
    index=False,
    encoding="utf-8-sig"
)

In [64]:
# =========================
# 7. 수동 조정 후 데이터 불러오기
# =========================
comment_final_df["group"] = comment_final_df["group"].fillna(0).astype(int)

print(comment_final_df["group"].value_counts().sort_index())
print(comment_final_df["FinalCategory"].isna().sum())

group
0    34571
1    38867
Name: count, dtype: int64
0


In [65]:
# =========================
# 8. group 0/1 없는 경우도 0으로 채우기 위한 기준표
# =========================

all_projects = comment_final_df["projectID"].dropna().unique()
all_groups = [0, 1]
all_categories = sorted(comment_final_df["FinalCategory"].dropna().unique())

base_project_group_category = pd.MultiIndex.from_product(
    [all_projects, all_groups, all_categories],
    names=["projectID", "group", "FinalCategory"]
).to_frame(index=False)

base_project_category = pd.MultiIndex.from_product(
    [all_projects, all_categories],
    names=["projectID", "FinalCategory"]
).to_frame(index=False)

In [66]:
# =========================
# 9-1. 전/후 group별 카운트
# =========================

project_group_category_count = (
    comment_final_df
    .groupby(["projectID", "group", "FinalCategory"])
    .size()
    .reset_index(name="count")
)

project_group_category_count = (
    base_project_group_category
    .merge(
        project_group_category_count,
        on=["projectID", "group", "FinalCategory"],
        how="left"
    )
)

project_group_category_count["count"] = project_group_category_count["count"].fillna(0).astype(int)

project_group_category_count.to_csv(
    "project_group_category_count.csv",
    index=False,
    encoding="utf-8-sig"
)

display(project_group_category_count.head())

,projectID,group,FinalCategory,count
0,5787,0,Campaign / Pledge,1
1,5787,0,Community Reaction,17
2,5787,0,Components & Production,3
3,5787,0,Gameplay / Rules / Content,29
4,5787,0,Language & Localization,10


In [67]:
# =========================
# 9-2. 전/후 group별 비율
# =========================

project_group_category_ratio = project_group_category_count.copy()

project_group_category_ratio["total_count"] = (
    project_group_category_ratio
    .groupby(["projectID", "group"])["count"]
    .transform("sum")
)

project_group_category_ratio["ratio"] = np.where(
    project_group_category_ratio["total_count"] > 0,
    project_group_category_ratio["count"] / project_group_category_ratio["total_count"],
    0
)

project_group_category_ratio.to_csv(
    "project_group_category_ratio.csv",
    index=False,
    encoding="utf-8-sig"
)

display(project_group_category_ratio.head())

,projectID,group,FinalCategory,count,total_count,ratio
0,5787,0,Campaign / Pledge,1,68,0.014706
1,5787,0,Community Reaction,17,68,0.250000
2,5787,0,Components & Production,3,68,0.044118
3,5787,0,Gameplay / Rules / Content,29,68,0.426471
4,5787,0,Language & Localization,10,68,0.147059


In [68]:
# =========================
# 9-3. 통합 카운트
# =========================

project_category_count = (
    comment_final_df
    .groupby(["projectID", "FinalCategory"])
    .size()
    .reset_index(name="count")
)

project_category_count = (
    base_project_category
    .merge(
        project_category_count,
        on=["projectID", "FinalCategory"],
        how="left"
    )
)

project_category_count["count"] = project_category_count["count"].fillna(0).astype(int)

project_category_count.to_csv(
    "project_category_count.csv",
    index=False,
    encoding="utf-8-sig"
)

display(project_category_count.head())

,projectID,FinalCategory,count
0,5787,Campaign / Pledge,15
1,5787,Community Reaction,24
2,5787,Components & Production,9
3,5787,Gameplay / Rules / Content,66
4,5787,Language & Localization,14


In [69]:
# =========================
# 9-4. 통합 비율
# =========================

project_category_ratio = project_category_count.copy()

project_category_ratio["total_count"] = (
    project_category_ratio
    .groupby("projectID")["count"]
    .transform("sum")
)

project_category_ratio["ratio"] = np.where(
    project_category_ratio["total_count"] > 0,
    project_category_ratio["count"] / project_category_ratio["total_count"],
    0
)

project_category_ratio.to_csv(
    "project_category_ratio.csv",
    index=False,
    encoding="utf-8-sig"
)

display(project_category_ratio.head())

,projectID,FinalCategory,count,total_count,ratio
0,5787,Campaign / Pledge,15,144,0.104167
1,5787,Community Reaction,24,144,0.166667
2,5787,Components & Production,9,144,0.062500
3,5787,Gameplay / Rules / Content,66,144,0.458333
4,5787,Language & Localization,14,144,0.097222


In [70]:
check1 = (
    project_group_category_count
    .groupby(["projectID", "group"])
    .size()
)

print(check1.describe())
display(check1.value_counts().sort_index())

count    946.0
mean       7.0
std        0.0
min        7.0
25%        7.0
50%        7.0
75%        7.0
max        7.0
dtype: float64


,count
7,946


In [71]:
ratio_check = (
    project_group_category_ratio
    .groupby(["projectID", "group"])["ratio"]
    .sum()
    .reset_index()
)

display(ratio_check.head())

,projectID,group,ratio
0,222,0,1.0
1,222,1,1.0
2,769,0,1.0
3,769,1,1.0
4,806,0,1.0


In [72]:
ratio_check["error"] = abs(ratio_check["ratio"] - 1)

display(
    ratio_check
    .sort_values("error", ascending=False)
    .head(20)
)

,projectID,group,ratio,error
469,3723,1,0.0,1.0
183,2275,1,0.0,1.0
844,6176,0,0.0,1.0
283,2816,1,0.0,1.0
329,2984,1,0.0,1.0
849,6194,1,0.0,1.0
188,2304,0,0.0,1.0
325,2977,1,0.0,1.0
643,4672,1,0.0,1.0
909,6867,1,0.0,1.0


In [73]:
# group 없는 프로젝트 0으로 채워졌는지 확인
single_group_projects = (
    comment_final_df
    .groupby("projectID")["group"]
    .nunique()
)

single_group_projects = single_group_projects[
    single_group_projects == 1
].index.tolist()

print("한쪽 group만 있는 프로젝트 수:", len(single_group_projects))

test_project = single_group_projects[0]

display(
    project_group_category_count[
        project_group_category_count["projectID"] == test_project
    ]
)

한쪽 group만 있는 프로젝트 수: 45


,projectID,group,FinalCategory,count
6538,2081,0,Campaign / Pledge,0
6539,2081,0,Community Reaction,0
6540,2081,0,Components & Production,0
6541,2081,0,Gameplay / Rules / Content,0
6542,2081,0,Language & Localization,0
6543,2081,0,Schedule / Updates,0
6544,2081,0,Shipping / Price / Tax,0
6545,2081,1,Campaign / Pledge,1
6546,2081,1,Community Reaction,0
6547,2081,1,Components & Production,0


In [74]:
overall_ratio_check = (
    project_category_ratio
    .groupby("projectID")["ratio"]
    .sum()
)

print(
    "최소:", overall_ratio_check.min(),
    "최대:", overall_ratio_check.max()
)

최소: 0.9999999999999999 최대: 1.0


In [75]:
print(
    "원본 댓글 수:",
    len(comment_final_df)
)

print(
    "group 카운트 합:",
    project_group_category_count["count"].sum()
)

print(
    "통합 카운트 합:",
    project_category_count["count"].sum()
)

원본 댓글 수: 73438
group 카운트 합: 73438
통합 카운트 합: 73438


In [76]:
ratio_check = (
    project_group_category_ratio
    .groupby(["projectID", "group"])
    .agg(
        ratio_sum=("ratio", "sum"),
        total_count=("count", "sum")
    )
    .reset_index()
)

ratio_check["status"] = np.where(
    ratio_check["total_count"] == 0,
    "empty_group",
    np.where(
        np.isclose(ratio_check["ratio_sum"], 1),
        "ok",
        "check"
    )
)

display(ratio_check["status"].value_counts())

display(
    ratio_check[ratio_check["status"] == "check"]
)

,count
status,
ok,901
empty_group,45


,projectID,group,ratio_sum,total_count,status


In [78]:
df3 = pd.read_csv("/content/project_group_category_ratio.csv")

In [79]:
df3.head()

,projectID,group,FinalCategory,count,total_count,ratio
0,5787,0,Campaign / Pledge,1,68,0.014706
1,5787,0,Community Reaction,17,68,0.250000
2,5787,0,Components & Production,3,68,0.044118
3,5787,0,Gameplay / Rules / Content,29,68,0.426471
4,5787,0,Language & Localization,10,68,0.147059


# wide

In [80]:
# 그룹 카운트
project_group_category_count_wide = (
    project_group_category_count
    .pivot_table(
        index=["projectID", "group"],
        columns="FinalCategory",
        values="count",
        fill_value=0
    )
    .reset_index()
)

project_group_category_count_wide.columns.name = None

display(project_group_category_count_wide.head())

project_group_category_count_wide.to_csv(
    "project_group_category_count_wide.csv",
    index=False,
    encoding="utf-8-sig"
)

,projectID,group,Campaign / Pledge,Community Reaction,Components & Production,Gameplay / Rules / Content,Language & Localization,Schedule / Updates,Shipping / Price / Tax
0,222,0,64.0,152.0,75.0,263.0,37.0,53.0,47.0
1,222,1,225.0,185.0,92.0,309.0,63.0,80.0,133.0
2,769,0,17.0,41.0,60.0,28.0,7.0,5.0,3.0
3,769,1,69.0,31.0,183.0,32.0,15.0,23.0,33.0
4,806,0,1.0,2.0,1.0,5.0,0.0,0.0,0.0


In [81]:
# 비율 wide
project_group_category_ratio_wide = (
    project_group_category_ratio
    .pivot_table(
        index=["projectID", "group"],
        columns="FinalCategory",
        values="ratio",
        fill_value=0
    )
    .reset_index()
)

project_group_category_ratio_wide.columns.name = None

display(project_group_category_ratio_wide.head())

project_group_category_ratio_wide.to_csv(
    "project_group_category_ratio_wide.csv",
    index=False,
    encoding="utf-8-sig"
)

,projectID,group,Campaign / Pledge,Community Reaction,Components & Production,Gameplay / Rules / Content,Language & Localization,Schedule / Updates,Shipping / Price / Tax
0,222,0,0.092619,0.219971,0.108538,0.380608,0.053546,0.076700,0.068017
1,222,1,0.206992,0.170193,0.084637,0.284269,0.057958,0.073597,0.122355
2,769,0,0.105590,0.254658,0.372671,0.173913,0.043478,0.031056,0.018634
3,769,1,0.178756,0.080311,0.474093,0.082902,0.038860,0.059585,0.085492
4,806,0,0.111111,0.222222,0.111111,0.555556,0.000000,0.000000,0.000000
